# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [176]:
from sklearn.datasets import  fetch_california_housing
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [177]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [178]:

df["Age"] = df["Age"].astype(float).astype("Int64")
df.shape
df["HomePlanet"]=df["HomePlanet"].replace("Europa","Earth")

df.head()

df["HomePlanet"].unique()
df = df.drop(columns=["Name","PassengerId"], errors="ignore")

In [179]:
#for numbers, takes the data that is most similar in other rows and it fill ins
imputer = KNNImputer(n_neighbors=5)
df_num = imputer.fit_transform(df[["Age", "RoomService", "Spa","ShoppingMall","FoodCourt","VRDeck"]])
df[["Age", "RoomService", "Spa","ShoppingMall","FoodCourt","VRDeck"]] = df_num
df[["Age", "RoomService", "Spa","ShoppingMall","FoodCourt","VRDeck"]].isna().sum()
#for booleans we use Simple Imputer

bools=["VIP","CryoSleep"]
imputer_bool=SimpleImputer(strategy="most_frequent")
df[bools]=imputer_bool.fit_transform(df[bools])
df[["VIP","CryoSleep"]].isna().sum()
cat_cols = ["HomePlanet", "Cabin", "Destination"]

imputer_cat = SimpleImputer(strategy="most_frequent")
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])
df["Cabin"] = df["Cabin"].str[0]
cat_cols = ["HomePlanet", "Cabin", "Destination", "VIP", "CryoSleep"]
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

df.head()

y=df["Transported"].astype(int)
X= df.drop(columns=["Transported"])

Now perform the same as before:
- Feature Scaling
- Feature Selection


**Perform Train Test Split**

In [180]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [181]:
normalizer = MinMaxScaler()

normalizer.fit(X_train)

MinMaxScaler()

In [182]:
X_train_norm = normalizer.transform(X_train)

X_test_norm = normalizer.transform(X_test)

In [183]:
X_train_norm = pd.DataFrame(X_train_norm, columns = X_train.columns)
X_train_norm.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Mars,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True,CryoSleep_True
0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.215190,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
2,0.443038,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,0.329114,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.164557,0.0,0.0,0.004897,0.000045,0.253098,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [184]:
X_test_norm = pd.DataFrame(X_test_norm, columns = X_test.columns)
X_test_norm.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Mars,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True,CryoSleep_True
0,0.430380,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.050633,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
2,0.316456,0.028617,0.001073,0.001143,0.055293,0.000492,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.151899,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.835443,0.000000,0.061316,0.000082,0.083586,0.002213,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [185]:
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth=20), n_estimators=100, max_samples = 1000)
bagging_reg.fit(X_train_norm, y_train)
pred = bagging_reg.predict(X_test_norm)

print("MAE", mean_absolute_error(pred, y_test))
print("RMSE", mean_squared_error(pred, y_test))
print("R2 score", bagging_reg.score(X_test_norm, y_test))

MAE 0.2785015609023648
RMSE 0.13818808785665182
R2 score 0.44721675683060347


- Random Forests

In [186]:
forest = RandomForestRegressor(n_estimators=100,max_depth=20)
forest.fit(X_train_norm, y_train)


RandomForestRegressor(max_depth=20)

In [187]:
pred = forest.predict(X_test_norm)

print("MAE", mean_absolute_error(pred, y_test))
print("RMSE", mean_squared_error(pred, y_test))
print("R2 score", forest.score(X_test_norm, y_test))

MAE 0.26846171483135584
RMSE 0.14329498619508757
R2 score 0.42678802183728526


- Gradient Boosting

In [188]:
gb_reg = GradientBoostingRegressor(max_depth=20, n_estimators=100)
gb_reg.fit(X_train_norm, y_train)
pred = gb_reg.predict(X_test_norm)

print("MAE", mean_absolute_error(pred, y_test))
print("RMSE", mean_squared_error(pred, y_test))
print("R2 score", gb_reg.score(X_test_norm, y_test))

MAE 0.26520738867852245
RMSE 0.1786459209393099
R2 score 0.28537638021118494


- Adaptive Boosting

In [189]:
ada_reg = AdaBoostRegressor(DecisionTreeRegressor(max_depth=20), n_estimators=100)
ada_reg.fit(X_train_norm, y_train)
pred = ada_reg.predict(X_test_norm)

print("MAE", mean_absolute_error(pred, y_test))
print("RMSE", mean_squared_error(pred, y_test))
print("R2 score", ada_reg.score(X_test_norm, y_test))

MAE 0.2495136914040277
RMSE 0.17606680633922764
R2 score 0.2956934151687707


Which model is the best and why?

In [ ]:
#Bagging Regressor achieves the lowest mean squared error of all the models, which indicates the best overall predictive performance. It also has the highest R², 
# meaning it explains the largest proportion of the variance in the target variable on the test set. Although its MAE is slightly higher than AdaBoost or Random 
# Forest, the difference is small, while the improvement in MSE and R² is clearly more substantial.